# Day 016 — Exercise 4: Stream Chat Turn

**Goal:** Implement `stream_chat_turn(history, user_input, model)` — stream the reply to stdout while collecting it, then return `(reply, new_history)`.

In [ ]:
import ollama
import io
import sys

In [ ]:
def stream_tokens(messages: list[dict], model: str = "llama3.2"):
    """Yield tokens one at a time from a streaming Ollama response."""
    response = ollama.chat(model=model, messages=messages, stream=True)
    for chunk in response:
        yield chunk["message"]["content"]

In [ ]:
def collect_stream(tokens) -> str:
    """Consume a token iterator and return the full reply as a string."""
    return "".join(tokens)

In [ ]:
def print_stream(tokens) -> None:
    """Print tokens to stdout as they arrive; final newline at end."""
    for token in tokens:
        print(token, end="", flush=True)
    print()

In [ ]:
def append_turn(history: list[dict], user_text: str, assistant_text: str) -> list[dict]:
    """Return a new history list with one user+assistant turn appended."""
    return history + [
        {"role": "user", "content": user_text},
        {"role": "assistant", "content": assistant_text},
    ]

## Your Implementation

In [ ]:
def stream_chat_turn(
    history: list[dict],
    user_input: str,
    model: str = "llama3.2",
) -> tuple[str, list[dict]]:
    """
    Stream the model reply to stdout and return (reply, new_history).

    Iterates over stream_tokens ONCE — printing each token AND collecting
    it. Calls append_turn to build the new history.
    """
    # TODO: build messages = history + [user message dict]
    # TODO: call stream_tokens(messages, model)
    # TODO: iterate ONCE — print each token AND append to parts list
    # TODO: print a final newline
    # TODO: assemble reply = "".join(parts)
    # TODO: return (reply, append_turn(history, user_input, reply))
    pass

## Check Your Work

In [ ]:
import io, sys

def _run_checks():
    total = 5
    passed = 0
    result = None   # guard so Checks 3-5 see a clear assertion failure if Check 2 fails

    # Check 1: stream_chat_turn is defined
    try:
        assert 'stream_chat_turn' in globals()
        passed += 1; print("✅ Check 1: stream_chat_turn is defined")
    except Exception as e:
        print(f"❌ Check 1: not defined — {e}")

    # Check 2: returns a tuple of length 2
    try:
        history = []
        old = sys.stdout; sys.stdout = io.StringIO()
        result = stream_chat_turn(history, "Say the word: yes")
        sys.stdout = old
        assert isinstance(result, tuple) and len(result) == 2, \
            f"expected tuple of length 2, got {type(result)}"
        passed += 1; print("✅ Check 2: returns a tuple of length 2")
    except Exception as e:
        sys.stdout = old
        print(f"❌ Check 2: return type — {e}")

    # Check 3: first element is a non-empty string
    try:
        assert isinstance(result, tuple), "Check 2 must pass first"
        reply, new_history = result
        assert isinstance(reply, str) and len(reply.strip()) > 0, \
            "first element should be a non-empty string"
        passed += 1; print("✅ Check 3: first element is a non-empty string")
    except Exception as e:
        print(f"❌ Check 3: reply string — {e}")

    # Check 4: new history has 2 more messages than input
    try:
        assert isinstance(result, tuple), "Check 2 must pass first"
        _, new_history = result
        assert isinstance(new_history, list), "second element must be a list"
        assert len(new_history) == 2, \
            f"empty input history → expected 2 messages (user+assistant), got {len(new_history)}"
        passed += 1; print("✅ Check 4: new history has 2 more messages than input")
    except Exception as e:
        print(f"❌ Check 4: history length — {e}")

    # Check 5: assistant message in history matches returned reply
    try:
        assert isinstance(result, tuple), "Check 2 must pass first"
        reply, new_history = result
        asst_msg = new_history[-1]
        assert asst_msg["role"] == "assistant", \
            f"last message role should be 'assistant', got {asst_msg['role']!r}"
        assert asst_msg["content"] == reply, \
            "history assistant content should match the returned reply"
        passed += 1; print("✅ Check 5: history assistant message matches reply")
    except Exception as e:
        print(f"❌ Check 5: history content — {e}")

    if passed == total:
        print("🎉 Exercise complete!")
    print(f"\nScore: {passed}/{total}")

_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
def stream_chat_turn(
    history: list[dict],
    user_input: str,
    model: str = "llama3.2",
) -> tuple[str, list[dict]]:
    messages = history + [{"role": "user", "content": user_input}]
    tokens = stream_tokens(messages, model)
    parts = []
    for token in tokens:
        print(token, end="", flush=True)
        parts.append(token)
    print()
    reply = "".join(parts)
    return reply, append_turn(history, user_input, reply)
```

</details>